# Data
Loads DESI spectra from MultimodalUniverse/desi on HuggingFace.
Schema: flux at item['spectrum']['flux'], redshift at item['Z'], 7781 pixels per spectrum.

In [ ]:
import os
os.environ["HF_DATASETS_VERBOSITY"] = "warning"

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, concatenate_datasets, load_from_disk
import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import config

In [ ]:
class DESISpectraDataset(Dataset):
    """PyTorch Dataset wrapping DESI spectra from UniverseTBD/mmu_desi_edr_sv3.

    Schema:
        item['spectrum']['flux']   — flux array (float32, ~7781 pixels)
        item['spectrum']['ivar']   — inverse variance; 0 = bad pixel
        item['spectrum']['mask']   — boolean bad-pixel mask; True = bad
        item['Z']                  — redshift (float32)
        item['ZWARN']              — bool; False = clean fit (we only load these)

    Args:
        hf_split: HuggingFace dataset split.
    """

    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Return a single normalized, padded spectrum and its redshift.

        Args:
            idx (int): Sample index.

        Returns:
            tuple: (flux (torch.Tensor, shape PADDED_LEN), redshift (torch.Tensor, scalar))
        """
        item = self.data[idx]
        spec = item['spectrum']
        flux = np.array(spec['flux'], dtype=np.float32)
        ivar = np.array(spec['ivar'], dtype=np.float32)
        mask = np.array(spec['mask'], dtype=bool)
        redshift = float(item['Z'])

        flux = np.nan_to_num(flux, nan=0.0, posinf=0.0, neginf=0.0)
        valid = (ivar > 0) & (~mask)
        flux[~valid] = 0.0
        flux = self._normalize(flux, valid)
        flux = self._pad(flux)

        return (
            torch.tensor(flux, dtype=torch.float32),
            torch.tensor(redshift, dtype=torch.float32),
        )

    def _normalize(self, flux, valid):
        """Apply median/MAD normalization using valid pixels only.

        Bad pixels (ivar==0 or mask==True) are excluded from statistics
        and set to zero after normalization.

        Args:
            flux (np.ndarray): Flux array with bad pixels already zeroed.
            valid (np.ndarray): Boolean mask; True where pixel is usable.

        Returns:
            np.ndarray: Normalized flux array with bad pixels zeroed.
        """
        if valid.any():
            valid_flux = flux[valid]
            median = np.median(valid_flux)
            mad = np.median(np.abs(valid_flux - median))
        else:
            median, mad = 0.0, 0.0
        flux = (flux - median) / (mad + 1e-8)
        flux[~valid] = 0.0
        return flux

    def _pad(self, flux):
        """Zero-pad or truncate flux to PADDED_LEN.

        Args:
            flux (np.ndarray): Flux array of arbitrary length.

        Returns:
            np.ndarray: Flux array of length PADDED_LEN.
        """
        padded = np.zeros(config.PADDED_LEN, dtype=np.float32)
        padded[:len(flux)] = flux[:config.PADDED_LEN]
        return padded

In [ ]:
def get_dataloaders(
    batch_size=config.BATCH_SIZE,
    n_train=180_000,
    n_val=20_000,
    chunk_size=50_000,
    max_raw=None,
    cache_path=None,
):
    """Load the UniverseTBD/mmu_desi_edr_sv3 dataset and return train/val DataLoaders.

    Streams the HuggingFace dataset in chunks of `chunk_size`, filtering each chunk
    to ZWARN==False (reliable redshift fits), until n_train + n_val clean examples
    are collected or the dataset is exhausted.

    num_workers=0 is intentional: %run-defined classes can't be pickled by
    multiprocessing worker subprocesses in a notebook environment.

    Args:
        batch_size (int): Batch size for both loaders.
        n_train (int): Number of training samples.
        n_val (int): Number of validation samples.
        chunk_size (int): Raw examples to load per iteration.
        max_raw (int or None): Hard cap on raw examples scanned. None = no limit.
        cache_path (str or None): Directory to save/load the clean dataset.
            None = no caching. If set and the directory exists, load from disk.
            If set and the directory does not exist, save after scanning.

    Returns:
        tuple: (train_loader, val_loader)

    Raises:
        RuntimeError: If the dataset is exhausted (or max_raw is reached) before
            enough clean examples are found, or if the cache size doesn't match
            n_train + n_val.
    """
    needed = n_train + n_val

    if cache_path is not None and os.path.exists(cache_path):
        print(f'Loading cached clean dataset from {cache_path}')
        clean = load_from_disk(cache_path)
        if len(clean) != needed:
            raise RuntimeError(
                f'Cache at {cache_path!r} has {len(clean):,} examples but '
                f'n_train + n_val = {needed:,}. '
                f'Delete the cache directory and re-run to rebuild it.'
            )
    else:
        clean_chunks = []
        n_clean = 0
        n_scanned = 0

        while n_clean < needed:
            if max_raw is not None and n_scanned >= max_raw:
                break

            start = n_scanned
            end = start + chunk_size
            if max_raw is not None:
                end = min(end, max_raw)

            try:
                raw_chunk = load_dataset(
                    'UniverseTBD/mmu_desi_edr_sv3',
                    split=f'train[{start}:{end}]',
                )
            except ValueError:
                break

            n_in_chunk = len(raw_chunk)
            if n_in_chunk == 0:
                break

            clean_chunk = raw_chunk.filter(lambda x: not x['ZWARN'])

            if len(clean_chunk) > 0:
                clean_chunks.append(clean_chunk)
            n_scanned += n_in_chunk
            n_clean += len(clean_chunk)

            print(f'  Scanned {n_scanned:,} raw  |  clean so far: {n_clean:,} / {needed:,}')

            if n_in_chunk < (end - start):
                break

        if n_clean < needed:
            raise RuntimeError(
                f'Dataset exhausted after scanning {n_scanned:,} raw examples. '
                f'Found {n_clean:,} clean examples but needed {needed:,}. '
                f'Lower n_train/n_val or raise max_raw.'
            )

        clean = concatenate_datasets(clean_chunks)
        clean = clean.select(range(needed))

        if cache_path is not None:
            print(f'Saving {needed:,} clean examples to {cache_path} ...')
            clean.save_to_disk(cache_path)
            print('  Saved.')

    clean = clean.shuffle(seed=42)

    train_ds = DESISpectraDataset(clean.select(range(n_train)))
    val_ds   = DESISpectraDataset(clean.select(range(n_train, needed)))

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=0, pin_memory=False, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=False, drop_last=False,
    )

    print(f'Train: {len(train_ds)} samples  Val: {len(val_ds)} samples')
    return train_loader, val_loader